# 🌙 06. FAISS Dense Vector Retrieval & Reference Candidate Matching

**Mission Context**: Real-time lunar orbiter global candidate retrieval from global reference basemaps.  
**Objectives**:
- Build FAISS Inner-Product/Cosine index on LunaDNA vectors.
- Perform top-K nearest neighbor searches under variable illumination.
- Evaluate Top-1, Top-5, and Top-10 Retrieval Accuracy.
- Profile query latencies and export retrieval audit logs.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.retrieval import FAISSRetrievalEngine

config = load_config()
vectors = np.load("outputs/lunadna/lunadna_vectors.npy")
df_dna = pd.read_csv("outputs/lunadna/lunadna_database.csv")
doc_ids = df_dna['file_name'].tolist()

faiss_engine = FAISSRetrievalEngine(dimension=256)
faiss_engine.build_index(vectors, doc_ids)
print(f"Indexed {len(doc_ids)} LunaDNA reference vectors in FAISS.")


In [ ]:
# Run Query Benchmark & Latency Profiling
latencies = []
top_k_hits = {1: 0, 5: 0, 10: 0}
N = len(vectors)

for i in range(N):
    q_vec = vectors[i]
    true_id = doc_ids[i]
    matched_ids, scores, lat_ms = faiss_engine.query(q_vec, top_k=10)
    latencies.append(lat_ms)

    for k in [1, 5, 10]:
        if true_id in matched_ids[:k]:
            top_k_hits[k] += 1

print(f"Top-1 Accuracy: {top_k_hits[1] / N * 100:.2f}%")
print(f"Top-5 Accuracy: {top_k_hits[5] / N * 100:.2f}%")
print(f"Top-10 Accuracy: {top_k_hits[10] / N * 100:.2f}%")
print(f"Mean FAISS Search Latency: {np.mean(latencies):.4f} ms per query")


In [ ]:
# Visualize Similarity Matrix & Latency Distribution
sim_matrix = np.dot(vectors, vectors.T)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(sim_matrix[:15, :15], ax=axes[0], cmap='viridis', annot=False)
axes[0].set_title("LunaDNA Pairwise Cosine Similarity Heatmap", fontweight='bold')
axes[0].set_xlabel("Tile Index")
axes[0].set_ylabel("Tile Index")

axes[1].hist(latencies, bins=15, color='#3B82F6', edgecolor='black', alpha=0.7)
axes[1].set_title("FAISS Query Latency Distribution (ms)", fontweight='bold')
axes[1].set_xlabel("Latency (ms)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/06_faiss_retrieval_benchmarks.png", dpi=300)
plt.show()
